# districtGroundTruth v9 (260920) — 행정경계 정답 대조

| 항목 | 내용 |
| --- | --- |
| 작성 일자 | 2026-09-20 |
| 선행 노트북 | `districtJudgeTest_v8_260920.ipynb` |
| 호출 API | V-World 2D 데이터 API (시군구 경계). **Tmap 호출 없음** |
| 산출물 | `./output9_260920/` |

## 1. 이 노트북이 답하는 것

v8에서 두 값을 비교했으나, **둘 다 실제 행정경계로 확인된 적이 없었다.**

| 비교 대상 | 정체 |
| --- | --- |
| 표본 군구 라벨 (`district_label`) | v6 표본을 만들 때 붙인 꼬리표. 인접군은 공업지대 폴리곤 기준, 분리군은 군구별 핫스팟 bounding box 기준 |
| 좌표 기반 군구 판정 (M1/M2/M3) | 출발지에서 가장 가까운 CCTV 핫스팟의 `source_district`를 취하는 방식 |

둘은 200쌍 중 39쌍에서 갈렸고, 그중 31쌍에서 최종 선정 핫스팟 2곳이 달라졌다.

본 노트북은 **V-World 행정경계 폴리곤을 정답지로 삼아**, 각 출발지가 실제로 어느 군구 안에
있는지 점-폴리곤 검사로 확인한다. 그 결과로 라벨과 좌표 판정 중 어느 쪽이 맞았는지 가린다.

## 2. 판정 기준 (결과를 보기 전에 고정한다)

주 지표는 **좌표 기반 군구 판정(M1)의 정확도**이다. 분모는 행정경계 폴리곤이 판정한 정답이다.

| M1 정확도 | 조치 |
| --- | --- |
| 95% 이상 | **좌표 판정 채택.** 서버에 M1 반영 |
| 표본 라벨 정확도보다 높으나 95% 미만 | 조건부 채택. 오차가 실제 경로에 미친 영향을 확인 후 결정 |
| 표본 라벨 정확도 이하 | **방법 A 전환.** 앱이 `district`를 전송하도록 수정 |

## 3. 실행 전 준비

```
./output6_260827/mixed_od_200.pkl                          (출발지 좌표)
./output8_260920/district_judge_compare_260920.csv         (v8 판정 결과)
.env 의 VWORLD_KEY                                          (v1에서 쓰던 키 그대로)
```

V-World 키에 도메인 제한이 걸려 있으면 `.env`에 `VWORLD_DOMAIN`도 넣는다.
인천 시군구 경계는 한 번만 받아 `./output9_260920/`에 저장하고, 이후 실행에서는 재사용한다.

## 4. 알아둘 점

- 인천에는 옹진군이 있으나 본 프로젝트의 핫스팟 데이터는 9개 군구(강화군 포함, 옹진군 제외)만
  다룬다. 옹진군으로 판정되는 출발지는 별도로 표시한다.
- 출발지가 어느 폴리곤에도 들어가지 않는 경우(해안선 오차 등)는 가장 가까운 폴리곤까지의
  거리를 함께 기록한다.
- `'동구'`는 `'남동구'`의 부분 문자열이므로, 군구 이름 매칭은 긴 이름부터 검사한다
  (v4_260729에서 발생했던 오분류 재발 방지).

In [1]:
# 셀 1: 라이브러리 import + V-World 인증키 로드

import os
import json
import math
import time
import pickle
from pathlib import Path
from collections import Counter

import requests
import numpy as np
import pandas as pd
from dotenv import load_dotenv

from shapely.geometry import shape, Point
from shapely.prepared import prep

load_dotenv(override=True)

VWORLD_KEY    = os.getenv("VWORLD_KEY")
VWORLD_DOMAIN = os.getenv("VWORLD_DOMAIN")      # 키에 도메인 제한이 없으면 None이어도 된다

if not VWORLD_KEY:
    raise RuntimeError(".env에 VWORLD_KEY가 없다.")

print("[v9 노트북 시작 — 행정경계 정답 대조]")
print(f"  V-World 키: {VWORLD_KEY[:4]}...{VWORLD_KEY[-4:]} ({len(VWORLD_KEY)}자)")
print(f"  domain 파라미터: {VWORLD_DOMAIN if VWORLD_DOMAIN else '미사용'}")
print("  Tmap 호출: 없음")

[v9 노트북 시작 — 행정경계 정답 대조]
  V-World 키: 8C07...3D4E (36자)
  domain 파라미터: 미사용
  Tmap 호출: 없음


In [2]:
# 셀 2: 경로 상수 + 입력 파일 점검

OD_PKL_PATH = Path("./output6_260827/mixed_od_200.pkl")
V8_CSV_PATH = Path("./output8_260920/district_judge_compare_260920.csv")

STAMP    = "260920"
V9_ROOT  = Path(f"./output9_{STAMP}")
V9_ROOT.mkdir(parents=True, exist_ok=True)
SIGG_GEOJSON = V9_ROOT / f"incheon_sigg_boundary_{STAMP}.geojson"

_missing = [p for p in (OD_PKL_PATH, V8_CSV_PATH) if not p.exists()]
if _missing:
    raise FileNotFoundError(
        "입력 파일이 없다:\n  " + "\n  ".join(str(p.resolve()) for p in _missing))

print("[입력 파일 확인]")
for p in (OD_PKL_PATH, V8_CSV_PATH):
    print(f"  {p}  ({p.stat().st_size/1024:.0f} KB)")
print(f"[출력 폴더] {V9_ROOT.resolve()}")
print(f"[경계 캐시] {SIGG_GEOJSON.name} "
      f"{'(존재 — 재사용)' if SIGG_GEOJSON.exists() else '(없음 — API로 수집)'}")

[입력 파일 확인]
  output6_260827\mixed_od_200.pkl  (1347 KB)
  output8_260920\district_judge_compare_260920.csv  (40 KB)
[출력 폴더] C:\Users\User\Documents\GitHub\ploggo-algorithm-test\output9_260920
[경계 캐시] incheon_sigg_boundary_260920.geojson (없음 — API로 수집)


In [3]:
# 셀 3: V-World에서 인천 시군구 경계 수집
#
# 2D 데이터 API / 시군구 경계 레이어 LT_C_ADSIGG_INFO
#   https://api.vworld.kr/req/data?service=data&request=GetFeature&data=LT_C_ADSIGG_INFO&...
# 응답 properties: sig_cd, sig_kor_nm, sig_eng_nm, full_nm
# 인천광역시 시군구 코드는 28로 시작한다.
#
# 캐시가 있으면 API를 호출하지 않는다.

VWORLD_DATA_URL = "https://api.vworld.kr/req/data"


def fetch_sigg(attr_filter, size=100):
    """LT_C_ADSIGG_INFO를 조회하여 features 리스트를 돌려준다. 실패 시 (None, 사유)."""
    params = {
        "service": "data", "request": "GetFeature", "version": "2.0",
        "data": "LT_C_ADSIGG_INFO", "key": VWORLD_KEY,
        "format": "json", "errorformat": "json",
        "size": str(size), "page": "1",
        "geometry": "true", "attribute": "true",
        "crs": "EPSG:4326",
        "attrFilter": attr_filter,
    }
    if VWORLD_DOMAIN:
        params["domain"] = VWORLD_DOMAIN

    try:
        r = requests.get(VWORLD_DATA_URL, params=params, timeout=30)
    except requests.exceptions.RequestException as e:
        return None, f"요청 실패: {type(e).__name__}"

    if r.status_code != 200:
        return None, f"HTTP {r.status_code}: {r.text[:200]}"

    try:
        body = r.json()["response"]
    except (ValueError, KeyError):
        return None, f"응답 파싱 실패: {r.text[:200]}"

    if body.get("status") != "OK":
        err = body.get("error", {})
        return None, f"status={body.get('status')} / {err.get('code')} {err.get('text')}"

    feats = body["result"]["featureCollection"]["features"]
    return feats, f"{len(feats)}건 수신"


if SIGG_GEOJSON.exists():
    with open(SIGG_GEOJSON, "r", encoding="utf-8") as f:
        sigg_fc = json.load(f)
    print(f"[캐시 재사용] {SIGG_GEOJSON.name} — {len(sigg_fc['features'])}건")
else:
    features, note = None, ""
    for attr in ("sig_cd:like:28", "full_nm:like:인천"):
        print(f"  시도: attrFilter={attr}")
        features, note = fetch_sigg(attr)
        print(f"    → {note}")
        if features:
            break

    if not features:
        raise RuntimeError(
            "V-World 시군구 경계 수집에 실패하였다.\n"
            "  1) VWORLD_KEY가 유효한지, 2D 데이터 API 사용 승인이 되어 있는지 확인한다.\n"
            "  2) 키에 도메인 제한이 있으면 .env에 VWORLD_DOMAIN을 추가한다.\n"
            f"  마지막 사유: {note}")

    sigg_fc = {"type": "FeatureCollection", "features": features}
    with open(SIGG_GEOJSON, "w", encoding="utf-8") as f:
        json.dump(sigg_fc, f, ensure_ascii=False)
    print(f"[저장] {SIGG_GEOJSON.resolve()}")

print("\n[수신 시군구]")
for ft in sigg_fc["features"]:
    p = ft["properties"]
    print(f"  {p.get('sig_cd'):<8} {p.get('sig_kor_nm'):<12} {p.get('full_nm')}")

  시도: attrFilter=sig_cd:like:28
    → 11건 수신
[저장] C:\Users\User\Documents\GitHub\ploggo-algorithm-test\output9_260920\incheon_sigg_boundary_260920.geojson

[수신 시군구]
  28125    제물포구         인천광역시 제물포구
  28710    강화군          인천광역시 강화군
  28720    옹진군          인천광역시 옹진군
  28155    영종구          인천광역시 영종구
  28177    미추홀구         인천광역시 미추홀구
  28185    연수구          인천광역시 연수구
  28200    남동구          인천광역시 남동구
  28237    부평구          인천광역시 부평구
  28245    계양구          인천광역시 계양구
  28275    서해구          인천광역시 서해구
  28290    검단구          인천광역시 검단구


In [4]:
# 셀 4: 폴리곤 변환 + 군구 영문 키 매핑
# '동구'가 '남동구'의 부분 문자열이므로 긴 이름부터 검사한다 (v4_260729 오분류 방지).

DISTRICT_MATCH_ORDER = [
    ("michuhol", ["미추홀구", "미추홀"]),
    ("namdong",  ["남동구",  "남동"]),
    ("bupyeong", ["부평구",  "부평"]),
    ("gyeyang",  ["계양구",  "계양"]),
    ("yeonsu",   ["연수구",  "연수"]),
    ("ganghwa",  ["강화군",  "강화"]),
    ("ongjin",   ["옹진군",  "옹진"]),     # 핫스팟 데이터에 없는 군구
    ("seogu",    ["서구"]),
    ("junggu",   ["중구"]),
    ("donggu",   ["동구"]),                # 반드시 '남동구'보다 뒤
]

PROJECT_DISTRICTS = {"michuhol", "namdong", "bupyeong", "gyeyang",
                     "yeonsu", "ganghwa", "seogu", "junggu", "donggu"}


def match_district(kor_name):
    s = str(kor_name)
    for eng, keys in DISTRICT_MATCH_ORDER:
        for k in keys:
            if k in s:
                return eng
    return None


sigg = []
for ft in sigg_fc["features"]:
    p = ft["properties"]
    eng = match_district(p.get("sig_kor_nm"))
    geom = shape(ft["geometry"])
    sigg.append({"eng": eng, "kor": p.get("sig_kor_nm"), "sig_cd": p.get("sig_cd"),
                 "geom": geom, "prep": prep(geom)})

print(f"[폴리곤 변환] {len(sigg)}개")
print(f"  {'영문키':<12} {'한글명':<12} {'코드':<8} {'프로젝트 대상':<10}")
print("-" * 50)
unmatched = []
for s in sigg:
    tag = "O" if s["eng"] in PROJECT_DISTRICTS else ("제외" if s["eng"] else "매칭실패")
    if s["eng"] is None:
        unmatched.append(s["kor"])
    print(f"  {str(s['eng']):<12} {str(s['kor']):<12} {str(s['sig_cd']):<8} {tag:<10}")

covered = {s["eng"] for s in sigg} & PROJECT_DISTRICTS
print(f"\n  프로젝트 9개 군구 중 수신됨: {len(covered)}개")
if len(covered) < 9:
    print(f"  [경고] 누락: {sorted(PROJECT_DISTRICTS - covered)}")
if unmatched:
    print(f"  [경고] 이름 매칭 실패: {unmatched}")

[폴리곤 변환] 11개
  영문키          한글명          코드       프로젝트 대상   
--------------------------------------------------
  None         제물포구         28125    매칭실패      
  ganghwa      강화군          28710    O         
  ongjin       옹진군          28720    제외        
  None         영종구          28155    매칭실패      
  michuhol     미추홀구         28177    O         
  yeonsu       연수구          28185    O         
  namdong      남동구          28200    O         
  bupyeong     부평구          28237    O         
  gyeyang      계양구          28245    O         
  None         서해구          28275    매칭실패      
  None         검단구          28290    매칭실패      

  프로젝트 9개 군구 중 수신됨: 6개
  [경고] 누락: ['donggu', 'junggu', 'seogu']
  [경고] 이름 매칭 실패: ['제물포구', '영종구', '서해구', '검단구']


In [5]:
# 셀 5: 200쌍 출발지에 대한 점-폴리곤 검사 (= 정답 산출)

with open(OD_PKL_PATH, "rb") as f:
    od_list = pickle.load(f)
v8 = pd.read_csv(V8_CSV_PATH, encoding="utf-8-sig")

print(f"[로드] OD {len(od_list)}쌍 / v8 결과 {len(v8)}행")

DEG_TO_KM = 111.0


def judge_by_boundary(origin):
    """출발지가 속한 군구를 행정경계 폴리곤으로 판정한다.
    Returns: (영문키 또는 None, 상태, 최근접 폴리곤까지 거리 km)
    """
    pt = Point(origin[1], origin[0])          # shapely는 (lon, lat)
    hits = [s for s in sigg if s["prep"].contains(pt)]
    if len(hits) == 1:
        return hits[0]["eng"], "포함", 0.0
    if len(hits) > 1:                          # 경계선 위 등 드문 경우
        return hits[0]["eng"], f"중복({len(hits)})", 0.0

    nearest = min(sigg, key=lambda s: s["geom"].distance(pt))
    return None, "미포함", nearest["geom"].distance(pt) * DEG_TO_KM


rows = []
t0 = time.time()
for od in od_list:
    truth, status, gap_km = judge_by_boundary(od["O"])
    rows.append({
        "od_id": od.get("od_id"),
        "truth": truth,
        "truth_status": status,
        "gap_to_poly_km": gap_km,
        "O_lat": od["O"][0], "O_lon": od["O"][1],
    })
truth_df = pd.DataFrame(rows)
print(f"[점-폴리곤 검사] 소요 {time.time()-t0:.1f}초")

print("\n[검사 상태]")
for k, v in truth_df["truth_status"].value_counts().items():
    print(f"  {k:<12} {v}쌍")

miss = truth_df[truth_df["truth_status"] == "미포함"]
if len(miss):
    print(f"\n  미포함 {len(miss)}쌍 — 최근접 폴리곤까지 거리")
    print(f"    중앙 {miss['gap_to_poly_km'].median()*1000:.0f}m  "
          f"최대 {miss['gap_to_poly_km'].max()*1000:.0f}m")
    print("    (해안선·경계 좌표 오차 범위면 무시 가능하다)")

ong = truth_df[truth_df["truth"] == "ongjin"]
if len(ong):
    print(f"\n  [주의] 옹진군으로 판정된 출발지 {len(ong)}쌍. "
          f"핫스팟 데이터에 없는 군구이므로 정확도 계산에서 제외한다.")

df = v8.merge(truth_df, on="od_id", how="left")
print(f"\n[병합 완료] {len(df)}행")

[로드] OD 200쌍 / v8 결과 200행
[점-폴리곤 검사] 소요 0.0초

[검사 상태]
  포함           166쌍
  미포함          34쌍

  미포함 34쌍 — 최근접 폴리곤까지 거리
    중앙 143m  최대 2669m
    (해안선·경계 좌표 오차 범위면 무시 가능하다)

[병합 완료] 200행


In [6]:
# 셀 6: 정확도 비교 — 표본 라벨 vs 좌표 판정

valid = df[df["truth"].isin(PROJECT_DISTRICTS)].copy()
print(f"[정확도 산출 대상] {len(valid)}/{len(df)}쌍 "
      f"(미포함·옹진군 제외)")

print()
print("=" * 72)
print("[정확도] 행정경계 정답과 얼마나 일치하는가")
print("=" * 72)
print(f"  {'비교 대상':<24} {'일치':>10} {'정확도':>9}")
print("-" * 48)
acc = {}
for col, label in (("district_label", "표본 군구 라벨"),
                   ("m1", "좌표 판정 M1 (최근접 1곳)"),
                   ("m2", "좌표 판정 M2 (최근접 10곳)"),
                   ("m3", "좌표 판정 M3 (반경 1km)")):
    hit = (valid[col] == valid["truth"]).sum()
    acc[col] = hit / len(valid) * 100
    print(f"  {label:<24} {hit:>6}/{len(valid):<3} {acc[col]:>8.1f}%")

print()
print("=" * 72)
print("[핵심] 라벨과 좌표 판정이 갈린 OD에서 누가 맞았나")
print("=" * 72)
split = valid[valid["district_label"] != valid["m1"]]
lab_win = int((split["district_label"] == split["truth"]).sum())
crd_win = int((split["m1"] == split["truth"]).sum())
both_wrong = len(split) - lab_win - crd_win
print(f"  두 값이 갈린 OD: {len(split)}쌍")
print(f"    표본 라벨이 정답     {lab_win:>3}쌍")
print(f"    좌표 판정이 정답     {crd_win:>3}쌍")
print(f"    둘 다 오답           {both_wrong:>3}쌍")

print()
print("=" * 72)
print("[경로 영향] 선정 핫스팟 2곳이 달라진 OD에서 누가 맞았나")
print("=" * 72)
diff = valid[~valid["same_m1"]]
print(f"  선정이 달라진 OD: {len(diff)}쌍")
if len(diff):
    print(f"    좌표 판정이 정답     {int((diff['m1']==diff['truth']).sum()):>3}쌍")
    print(f"    표본 라벨이 정답     {int((diff['district_label']==diff['truth']).sum()):>3}쌍")

print()
print("[좌표 판정이 틀린 OD 상세]")
wrong = valid[valid["m1"] != valid["truth"]]
if len(wrong):
    cols = ["od_id", "group", "truth", "district_label", "m1", "nearest_m", "same_m1"]
    print(wrong[cols].to_string(index=False))
else:
    print("  없음")

[정확도 산출 대상] 115/200쌍 (미포함·옹진군 제외)

[정확도] 행정경계 정답과 얼마나 일치하는가
  비교 대상                            일치       정확도
------------------------------------------------
  표본 군구 라벨                     94/115     81.7%
  좌표 판정 M1 (최근접 1곳)           104/115     90.4%
  좌표 판정 M2 (최근접 10곳)          102/115     88.7%
  좌표 판정 M3 (반경 1km)           103/115     89.6%

[핵심] 라벨과 좌표 판정이 갈린 OD에서 누가 맞았나
  두 값이 갈린 OD: 23쌍
    표본 라벨이 정답       6쌍
    좌표 판정이 정답      16쌍
    둘 다 오답             1쌍

[경로 영향] 선정 핫스팟 2곳이 달라진 OD에서 누가 맞았나
  선정이 달라진 OD: 17쌍
    좌표 판정이 정답      12쌍
    표본 라벨이 정답       5쌍

[좌표 판정이 틀린 OD 상세]
od_id     group    truth district_label      m1  nearest_m  same_m1
 V002  adjacent michuhol       michuhol  yeonsu 923.740909    False
 V007  adjacent bupyeong          seogu   seogu 557.251686     True
 V027  adjacent michuhol         junggu  junggu 440.836894     True
 V083  adjacent michuhol       michuhol  yeonsu 437.977899    False
 V092  adjacent michuhol       michuhol  yeonsu 794.658246    False
 S

In [7]:
# 셀 7: 최종 판정 + 저장

a_m1, a_lab = acc["m1"], acc["district_label"]

if a_m1 >= 95:
    verdict = "좌표 판정 채택 — 서버에 M1 반영"
elif a_m1 > a_lab:
    verdict = "조건부 채택 — 오차가 경로에 미친 영향 확인 후 결정"
else:
    verdict = "방법 A 전환 — 앱이 district를 전송하도록 수정"

print("=" * 72)
print("[최종 판정 — 셀 0의 사전 기준 적용]")
print("=" * 72)
print(f"  좌표 판정 M1 정확도   {a_m1:5.1f}%")
print(f"  표본 군구 라벨 정확도 {a_lab:5.1f}%")
print(f"\n  → {verdict}")

best_coord = max(("m1", "m2", "m3"), key=lambda c: acc[c])
if best_coord != "m1" and acc[best_coord] - a_m1 >= 1.0:
    print(f"\n  참고: {best_coord.upper()}가 {acc[best_coord]:.1f}%로 더 높다. "
          f"차이가 크면 해당 방식 채택을 검토한다.")
else:
    print(f"\n  참고: M1~M3 정확도 차이가 크지 않다. 가장 단순한 M1을 쓴다.")

out = V9_ROOT / f"ground_truth_compare_{STAMP}.csv"
df.to_csv(out, index=False, encoding="utf-8-sig")
print(f"\n[저장] {out.resolve()}")
print(f"[저장] {SIGG_GEOJSON.resolve()}  (경계 폴리곤 캐시)")

[최종 판정 — 셀 0의 사전 기준 적용]
  좌표 판정 M1 정확도    90.4%
  표본 군구 라벨 정확도  81.7%

  → 조건부 채택 — 오차가 경로에 미친 영향 확인 후 결정

  참고: M1~M3 정확도 차이가 크지 않다. 가장 단순한 M1을 쓴다.

[저장] C:\Users\User\Documents\GitHub\ploggo-algorithm-test\output9_260920\ground_truth_compare_260920.csv
[저장] C:\Users\User\Documents\GitHub\ploggo-algorithm-test\output9_260920\incheon_sigg_boundary_260920.geojson  (경계 폴리곤 캐시)
